# Held-out confirmation of 35 metabolite interactions


In [ ]:
import os, json, time, warnings
from pathlib import Path
os.environ['CHD_RUN_ID'] = os.environ.get('CHD_PRIMARY_RUN_ID', 'final_rerun_2026')
os.environ['CHD_COHORT_ID']='prs_raw106_complete'
os.environ['CHD_CPU_WORKERS']=os.environ.get('CHD_TEST_INTERACTION_WORKERS','12')
os.environ['CHD_INNER_THREADS']='1'
shared = Path(os.environ.get("CHD_SHARED_NOTEBOOK", str(Path.cwd() / "00_shared_config.ipynb")))
if not shared.is_file():
    raise FileNotFoundError(shared)
get_ipython().run_line_magic('run',f'"{shared}"')
print('Loaded:',shared,'| run=',RUN_ID,'| cohort=',CHD_COHORT_ID,flush=True)


In [ ]:
import numpy as np
import pandas as pd
EXPECTED_N = 35
source_filename = (
    "result6_prs_metabolite_interactions_discovery_replication.csv"
)


main_results_dir = Path(TABLE_DIR).resolve().parent

preferred_source = (
    main_results_dir
    / "result6_exploratory_metabolites"
    / "tables"
    / source_filename
)

legacy_source = Path(TABLE_DIR).resolve() / source_filename

if preferred_source.is_file():
    source = preferred_source

elif legacy_source.is_file():
    source = legacy_source

else:

    matches = sorted({
        path.resolve()
        for path in main_results_dir.rglob(source_filename)
        if path.is_file()
    })

    if len(matches) == 0:
        raise FileNotFoundError(
            "Result 6 training and validation interaction results were not found.\n"
            f"Primary results directory: {main_results_dir}\n"
            f"Expected file: {source_filename}\n"

        )

    if len(matches) > 1:
        match_text = "\n".join(
            f"  - {path}" for path in matches
        )
        raise RuntimeError(
            "Multiple matching Result 6 files in the primary run. "
            "Matching paths:\n"
            f"{match_text}"
        )

    source = matches[0]

source = Path(source).resolve()

print("Locked Result 6 discovery/validation source:")
print(source)
d=pd.read_csv(source)
required={'Metabolite','Metabolite_Label','coef','p','FDR','Replication_Beta','Replication_P','Replication_FDR','Direction_Replicated','Fit_Error','Replication_Fit_Error'}
if required.difference(d): raise RuntimeError('Missing columns: '+','.join(sorted(required.difference(d))))
if len(d)!=106 or d.Metabolite.astype(str).nunique()!=106: raise RuntimeError('The Result 6 interaction table must contain 106 unique metabolites.')
for c in ['coef','p','FDR','Replication_Beta','Replication_P','Replication_FDR']: d[c]=pd.to_numeric(d[c],errors='coerce')
fit_ok=d.Fit_Error.fillna('').astype(str).str.strip().eq('') & d.Replication_Fit_Error.fillna('').astype(str).str.strip().eq('')
same=d.Direction_Replicated.astype(str).str.lower().eq('true')
cand=d.loc[fit_ok & same & d.FDR.lt(.05) & d.Replication_FDR.lt(.05)].copy()
cand['Metabolite']=cand.Metabolite.astype(str)
cand=cand.sort_values(['Replication_FDR','FDR','Metabolite']).reset_index(drop=True)
if len(cand)!=EXPECTED_N: raise RuntimeError(f'Expected 35 candidates, found {len(cand)}')
if not set(cand.Metabolite).issubset(set(map(str,nmr106_cols))): raise RuntimeError('Candidates fall outside the Raw106 panel.')
OUT=Path(RESULTS_OUTPUT_DIR)/'result6_test_confirmation_35_v1'
CKPT=OUT/'checkpoints'; OUT_TABLE=OUT/'tables'; OUT_FIG=OUT/'figures'
for p in [OUT,CKPT,OUT_TABLE,OUT_FIG]: p.mkdir(parents=True,exist_ok=True)
lock_table=cand[['Metabolite','Metabolite_Label','coef','p','FDR','Replication_Beta','Replication_P','Replication_FDR']].rename(columns={'coef':'Train_Beta','p':'Train_P','FDR':'Train_FDR_BH_106','Replication_Beta':'Validation_Beta','Replication_P':'Validation_P','Replication_FDR':'Validation_FDR_BH_106'})
lock_table.to_csv(OUT_TABLE/'candidate_lock_35_before_test.csv',index=False)
lock_payload={'specification_id':'result6_test_confirmation_35_v1','pipeline_manifest_sha256':current_pipeline_sha256(),'source_sha256':sha256_file(source),'candidate_rule':'Train BH-FDR<0.05 AND validation BH-FDR<0.05 AND same direction','candidate_fields':lock_table.Metabolite.tolist(),'model':'Clinical_plus_PRS + one metabolite + PRS x metabolite','preprocessing':'training-only imputation and StandardScaler','estimator':'unpenalized Cox PH with robust SE','test_family':'35 two-sided P values; BH including failures as P=1','confirmation_rule':'test BH-FDR<0.05 and same direction in train, validation and test','external_replication':False}
lock_sig=canonical_json_digest(lock_payload); lock_path=OUT/'CANDIDATE_LOCK_BEFORE_TEST.json'
if lock_path.exists():
    old=json.loads(lock_path.read_text(encoding='utf-8'))
    if old.get('signature')!=lock_sig or old.get('signature_payload')!=lock_payload: raise RuntimeError('The existing candidate lock differs from the current specification.')
    print('RESUME identical candidate lock:',lock_path,flush=True)
else:
    atomic_write_json({'locked_at':str(pd.Timestamp.now()),'signature':lock_sig,'signature_payload':lock_payload},lock_path)
    print('LOCKED BEFORE TEST FITTING:',lock_path,flush=True)
display(lock_table)


In [ ]:
df_train,df_val,df_test,final_feature_sets,pipeline_manifest=validate_pipeline_artifacts(load_frames=True)
if current_pipeline_sha256()!=lock_payload['pipeline_manifest_sha256']: raise RuntimeError('The pipeline manifest changed after candidate selection.')
if CHD_COHORT_ID!='prs_raw106_complete' or ANALYSIS_ROLE!='primary': raise RuntimeError('The primary cohort is required.')
base_features=list(final_feature_sets['Clinical_plus_PRS']); fields=lock_table.Metabolite.tolist()
if PRS_COL not in base_features or set(base_features).intersection(fields): raise RuntimeError('Unexpected base covariates.')
bt,bv,bx,base_cols=prepare_X_train_val_test(df_train,df_val,df_test,base_features,scale=True)
mt,mv,mx,nmr_cols=prepare_X_train_val_test(df_train,df_val,df_test,fields,scale=True)
base_cols=list(base_cols); nmr_cols=list(nmr_cols)
if nmr_cols!=fields: raise RuntimeError('Preprocessing changed the order of the 35 candidates.')
Xbase=np.ascontiguousarray(bx.to_numpy(float)); Xnmr=np.ascontiguousarray(mx[fields].to_numpy(float))
tt=np.ascontiguousarray(df_test.Time.to_numpy(float)); te=np.ascontiguousarray(df_test.Event.to_numpy(np.int8))
prs_idx=base_cols.index(PRS_COL); field_idx={f:i for i,f in enumerate(fields)}
if not np.isfinite(Xbase).all() or not np.isfinite(Xnmr).all(): raise RuntimeError('Non-finite values remain after preprocessing.')

audit=[]
for f in fields[:3]:
    _,_,old,cols=prepare_X_train_val_test(df_train,df_val,df_test,list(dict.fromkeys(base_features+[f])),scale=True)
    reuse=np.column_stack([Xbase,Xnmr[:,field_idx[f]]]); diff=float(np.max(np.abs(old.to_numpy(float)-reuse)))
    if list(cols)!=base_cols+[f] or diff>1e-10: raise RuntimeError(f'Preprocessing equivalence check failed: {f}, {diff}')
    audit.append({'Metabolite':f,'Test_Max_Absolute_Difference':diff,'Pass':True})
pd.DataFrame(audit).to_csv(OUT_TABLE/'preprocessing_equivalence_audit.csv',index=False)
design={'lock_signature':lock_sig,'manifest':current_pipeline_sha256(),'test_eid':eid_digest(df_test),'test_outcome':dataframe_digest(df_test,['Eid','Time','Event']),'base_columns':base_cols,'fields':fields}
design_sig=canonical_json_digest(design)
print(f'Test ready: N={len(df_test):,}; events={int(te.sum()):,}; preprocessing audit=PASS',flush=True)


In [ ]:
from joblib import Parallel,delayed
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceWarning
workers=min(max(1,int(os.environ.get('CHD_TEST_INTERACTION_WORKERS',str(CPU_WORKERS)))),EXPECTED_N)
def cp(f): return CKPT/f'{safe_filename(str(f))}.json'
def cp_payload(f): return {'spec':'result6_test_confirmation_35_v1','field':str(f),'lock':lock_sig,'design':design_sig,'penalizer':0.0,'robust':True,'ties':'efron'}
def fit_one(f):
    f=str(f); path=cp(f); payload=cp_payload(f)
    if path.exists():
        try:
            saved=json.loads(path.read_text(encoding='utf-8'))
            if saved.get('signature_payload')==payload and saved.get('signature')==canonical_json_digest(payload) and not str(saved.get('row',{}).get('Test_Fit_Error','')).strip():
                print('RESUME',f,flush=True); return saved['row']
        except Exception: pass
    start=time.time(); label=lock_table.loc[lock_table.Metabolite.eq(f),'Metabolite_Label'].iloc[0]
    m=Xnmr[:,field_idx[f]]; interaction=Xbase[:,prs_idx]*m
    frame=pd.DataFrame(np.column_stack([Xbase,m,interaction]),columns=base_cols+[f,'PRS_x_Metabolite'])
    frame.insert(0,'Event',te); frame.insert(0,'Time',tt)
    row={'Metabolite':f,'Metabolite_Label':label,'Test_N':len(frame),'Test_Event_N':int(te.sum()),'Test_Fit_Error':''}
    try:
        model=CoxPHFitter(penalizer=0.0,l1_ratio=0.0)
        with warnings.catch_warnings():
            warnings.simplefilter('error',ConvergenceWarning)
            model.fit(frame,duration_col='Time',event_col='Event',robust=True,batch_mode=True,show_progress=False)
        z=model.summary.loc['PRS_x_Metabolite']
        row.update({'Test_Beta':float(z['coef']),'Test_SE':float(z['se(coef)']),'Test_P':float(z['p']),'Test_HR':float(z['exp(coef)']),'Test_HR_CI_Lower':float(z['exp(coef) lower 95%']),'Test_HR_CI_Upper':float(z['exp(coef) upper 95%'])})
        atomic_write_json({'signature':canonical_json_digest(payload),'signature_payload':payload,'row':row},path); status='OK'
    except Exception as e:
        row.update({'Test_Beta':np.nan,'Test_SE':np.nan,'Test_P':1.0,'Test_HR':np.nan,'Test_HR_CI_Lower':np.nan,'Test_HR_CI_Upper':np.nan,'Test_Fit_Error':repr(e)}); status='ERROR'
    print(f'DONE {label} ({f}) | {status} | {(time.time()-start)/60:.1f} min',flush=True); return row
print(f'START held-out confirmation: 35 tasks | {workers} workers',flush=True); start=time.time()
rows=Parallel(n_jobs=workers,backend='loky',verbose=10,max_nbytes='10M',mmap_mode='r',pre_dispatch=workers,batch_size=1)(delayed(fit_one)(f) for f in fields)
test=pd.DataFrame(rows); print(f'ALL FITS RETURNED | {(time.time()-start)/60:.1f} min',flush=True)


In [ ]:
def bh(p):
    p=np.where(np.isfinite(np.asarray(p,float)),p,1.0); o=np.argsort(p); r=p[o]
    a=np.minimum.accumulate((r*len(r)/np.arange(1,len(r)+1))[::-1])[::-1]; out=np.empty_like(a); out[o]=np.clip(a,0,1); return out
if len(test)!=35 or test.Metabolite.astype(str).nunique()!=35: raise RuntimeError('Test results must contain exactly the 35 locked candidates.')
res=lock_table.merge(test,on=['Metabolite','Metabolite_Label'],validate='one_to_one')
res['Test_FDR_BH_35']=bh(res.Test_P.fillna(1.0))
res['Test_P_Bonferroni_35']=np.minimum(res.Test_P.fillna(1.0)*35,1.0)
res['Three_Split_Same_Direction']=np.sign(res.Train_Beta).eq(np.sign(res.Validation_Beta)) & np.sign(res.Train_Beta).eq(np.sign(res.Test_Beta))
res['Test_Nominal_Directional']=res.Test_P.lt(.05) & res.Three_Split_Same_Direction
res['Heldout_Internal_Confirmation_BH']=res.Test_FDR_BH_35.lt(.05) & res.Three_Split_Same_Direction & res.Test_Fit_Error.fillna('').astype(str).str.strip().eq('')
res=res.sort_values(['Heldout_Internal_Confirmation_BH','Test_FDR_BH_35','Metabolite_Label'],ascending=[False,True,True]).reset_index(drop=True)
result_path=OUT_TABLE/'result6_prs_metabolite_interactions_test_confirmation_35.csv'; res.to_csv(result_path,index=False)
errors=int(res.Test_Fit_Error.fillna('').astype(str).str.strip().ne('').sum())
summary={'status':'SUCCESS' if errors==0 else 'INCOMPLETE','candidate_n':35,'fit_error_n':errors,'test_directional_nominal_n':int(res.Test_Nominal_Directional.sum()),'test_directional_bh_fdr_0.05_n':int(res.Heldout_Internal_Confirmation_BH.sum()),'test_n':len(df_test),'test_event_n':int(te.sum()),'candidate_lock_signature':lock_sig,'terminology':'held-out internally confirmed; not external replication','result_table':str(result_path)}
atomic_write_json(summary,OUT/('SUCCESS.json' if errors==0 else 'INCOMPLETE.json'))
print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True); display(res)
if errors: raise RuntimeError(f'{errors} models failed during confirmation.')


In [ ]:
from matplotlib.ticker import FuncFormatter
plot=res.sort_values(['Test_Beta','Metabolite_Label']).reset_index(drop=True)
colors=np.where(plot.Heldout_Internal_Confirmation_BH,np.where(plot.Test_Beta.ge(0),CNS['orange'],CNS['blue']),CNS['gray'])
labels=plot.Metabolite_Label.astype(str)+np.where(plot.Heldout_Internal_Confirmation_BH,'  *','')
fig,ax=plt.subplots(figsize=(9.2,12.2)); y=np.arange(len(plot))
for i,row in plot.iterrows():
    if np.isfinite(row.Test_HR): ax.errorbar(row.Test_HR,i,xerr=[[max(0,row.Test_HR-row.Test_HR_CI_Lower)],[max(0,row.Test_HR_CI_Upper-row.Test_HR)]],fmt='o',color=colors[i],ecolor=colors[i],capsize=2.5,markersize=5.5)
ax.axvline(1,color=CNS['black'],ls=':',lw=1); ax.set_xscale('log'); ax.xaxis.set_major_formatter(FuncFormatter(lambda x,_:f'{x:.2f}'))
ax.set_yticks(y); ax.set_yticklabels(labels); ax.set_ylim(-.7,len(plot)-.3)
ax.set_xlabel('Test-set multiplicative interaction HR per 1-SD PRS × 1-SD metabolite (95% CI)')
ax.set_title('Held-out internal confirmation of 35 prelocked interactions',loc='left',fontweight='bold'); ax.grid(axis='x',color=CNS['light_gray'],lw=.7); fig.tight_layout()
stem=OUT_FIG/'supplement_result6_test_confirmation_35_forest'
for ext in ['png','pdf','svg']: fig.savefig(str(stem)+'.'+ext,dpi=600,bbox_inches='tight')
plt.show(); print('Saved:',str(stem)+'.png',flush=True)
